<div style="background-color: #c62cce ; padding: 20px; border-radius: 10px;">

<h1 style="color: black; text-align: center;">
    Projet ML : Diabetes Patient Readmission Prediction Using Machine Learning 🩺
</h1>

<h3 style="color: black; text-align: center;">
    Notebook Model training
</h3>

<h5 style="color: black; text-align: center;">
    Ovia Chanemouganandam & Sandrine Daniel – DIA 4
</h5>

</div>


Import libraries

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import accuracy_score, recall_score, precision_score, f1_score, roc_auc_score, classification_report
from sklearn.linear_model import LogisticRegression 
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from imblearn.over_sampling import SMOTE
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.model_selection import cross_val_score
from sklearn.model_selection import GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import BaggingClassifier, VotingClassifier, StackingClassifier
from catboost import CatBoostClassifier


## **Loading data from preprocessing**

In [ ]:

df_clean = pd.read_csv("data/df_clean.csv")
df_scaled = pd.read_csv("data/df_scaled.csv")

X_train = pd.read_csv("data/X_train.csv")
X_test = pd.read_csv("data/X_test.csv")

y_train = pd.read_csv("data/y_train.csv")["readmitted"]
y_test = pd.read_csv("data/y_test.csv")["readmitted"]

X_train_scaled = pd.read_csv("data/X_train_scaled.csv")
X_test_scaled = pd.read_csv("data/X_test_scaled.csv")

## **Helper function for metrics**

In [ ]:
def evaluate_model_multiclass(model, X_test, y_test):
    """
    Evaluate a multiclass classifier on standard metrics.
    """
    y_pred = model.predict(X_test)

    results = {
        "Accuracy": accuracy_score(y_test, y_pred),
        "Precision_macro": precision_score(y_test, y_pred, average="macro"),
        "Recall_macro": recall_score(y_test, y_pred, average="macro"),
        "F1_macro": f1_score(y_test, y_pred, average="macro"),
        "F1_weighted": f1_score(y_test, y_pred, average="weighted"),
    }

    # Optional: multiclass ROC-AUC (only if model has predict_proba)
    if hasattr(model, "predict_proba"):
        y_proba = model.predict_proba(X_test)  # shape (n_samples, n_classes)
        results["ROC-AUC_ovr_macro"] = roc_auc_score(
            y_test, y_proba,
            multi_class="ovr",
            average="macro"
        )

    return results


# <font color='#c62cce'> <b>1. Creation of the first models  </b></font>

### **a) Logistic regression**

In [ ]:
log_reg = LogisticRegression(
    class_weight="balanced",
    multi_class="multinomial",   # tells it we have several classes
    solver="lbfgs",              # supports multinomial
    max_iter=1000
)

log_reg.fit(X_train, y_train)  # use your imputed data
results_logreg = evaluate_model_multiclass(log_reg, X_test, y_test)
results_logreg

## **b) Decision Tree**

In [ ]:
dt = DecisionTreeClassifier(class_weight='balanced', random_state=42)
dt.fit(X_train, y_train)

results_dt = evaluate_model_multiclass(dt, X_test, y_test)
results_dt

## **c) RandomForestClassifier**

In [ ]:
rf = RandomForestClassifier(class_weight='balanced', random_state=42)
rf.fit(X_train, y_train)

results_rf = evaluate_model_multiclass(rf, X_test, y_test)
results_rf

## **d) KNeighborsClassifier**

In [ ]:
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_scaled, y_train)

results_knn = evaluate_model_multiclass(knn, X_test_scaled, y_test)
results_knn

## **e) XGBoost**

In [ ]:
X_train_xgb = X_train.rename(columns=lambda c: c.replace("[","").replace("]","").replace("(","").replace(")","").replace("<","").replace(">",""))
X_test_xgb = X_test.rename(columns=lambda c: c.replace("[","").replace("]","").replace("(","").replace(")","").replace("<","").replace(">",""))

In [ ]:
xgb_base = XGBClassifier(
    objective='multi:softprob',
    num_class=3,
    eval_metric='mlogloss',
    tree_method='hist',
    random_state=42
)

xgb_base.fit(X_train_xgb, y_train)

results_xgb_base = evaluate_model_multiclass(xgb_base, X_test_xgb, y_test)
results_xgb_base

# <font color='#c62cce'> <b>2. Handling class imbalance </b></font>

We first handled class imbalance using class_weight = balanced now we will try SMOTE only for models that benefit from it  

In [ ]:
# Apply SMOTE (only on training set)

smote = SMOTE(random_state=42)

X_train_sm, y_train_sm = smote.fit_resample(X_train, y_train)

print("Before SMOTE:\n", y_train.value_counts())
print("\nAfter SMOTE:\n", y_train_sm.value_counts()) 

## **a) Logistic Regression with SMOTE**

Here we use a pipeline: StandardScaler + LogisticRegression.

In [ ]:
# ============================================
# Logistic Regression with SMOTE (pipeline)
# ============================================

pipe_lr_smote = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        multi_class="multinomial",
        max_iter=1000,
        random_state=42
    ))
])

pipe_lr_smote.fit(X_train_sm, y_train_sm)

results_lr_smote = evaluate_model_multiclass(pipe_lr_smote, X_test, y_test)
results_lr_smote

For Logistic Regression, applying SMOTE decreased the F1-macro score.
This is expected because Logistic Regression is a linear classifier that does not model synthetic minority samples well.
SMOTE introduced interpolated points that distorted the decision boundary and degraded generalization.
Therefore, we kept the non-SMOTE version as the best-performing configuration

## **b) Decision Tree with SMOTE**

Tree models don’t need scaling, so no pipeline here

In [ ]:
# ============================================
# Decision Tree with SMOTE
# ============================================

dt_smote = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42,
    criterion="gini"  # you can later tune gini vs entropy
)

dt_smote.fit(X_train_sm, y_train_sm)

results_dt_smote = evaluate_model_multiclass(dt_smote, X_test, y_test)
results_dt_smote

Applying SMOTE to the Decision Tree degraded the model’s performance.
This behavior is expected because Decision Trees are very sensitive to noisy synthetic samples created by SMOTE.
Since the minority classes in this dataset are highly noisy, SMOTE amplifies this noise and leads the tree to overfit unrealistic synthetic points.
As a result, the Decision Tree without SMOTE achieves a better F1-macro score than the SMOTE version.
Therefore, for tree-based models, class_weight='balanced' (or no imbalance correction) is generally more effective than SMOTE

## **c) KNN with SMOTE + Pipeline**

For KNN we must scale, and it benefits from SMOTE, so we use an Pipeline

In [ ]:

# ============================================
# KNN with SMOTE + scaling (optional)
# ============================================

pipe_knn_smote = ImbPipeline([
    ("smote", SMOTE(random_state=42)),
    ("scaler", StandardScaler()),
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

pipe_knn_smote.fit(X_train, y_train)

results_knn_smote = evaluate_model_multiclass(pipe_knn_smote, X_test, y_test)
results_knn_smote




KNN performs worse when trained on SMOTE-resampled data.
This is expected because KNN is highly sensitive to the geometric structure of the data.
SMOTE generates synthetic minority-class samples by interpolating between existing points, which distorts the true distance relationships in the feature space.
As a result, the KNN decision boundaries become less meaningful, leading to lower F1-macro and ROC-AUC scores.
Therefore, the version without SMOTE provides more reliable performance for KNN on this dataset

# <font color='#c62cce'> <b>3. Stratified K-fold Cross-validation  </b></font>

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

def cv_f1_macro(model, X, y, model_name="Model"):
    """
    Runs cross-validation with F1-macro and prints the mean score.
    """
    scores = cross_val_score(
        model,
        X,
        y,
        scoring="f1_macro",
        cv=cv,
        n_jobs=-1
    )
    print(f"{model_name} — CV F1-macro: {scores.mean():.4f} ± {scores.std():.4f}")
    return scores.mean()

cv_f1_macro(log_reg, X_train, y_train, "Logistic Regression (no SMOTE)")



cv_f1_macro(dt, X_train, y_train, "Decision tree (baseline)")



cv_f1_macro(xgb_base, X_train_xgb, y_train, "XGBoost (baseline)")


Cross-validation scores were slightly lower than the single train/test split result, which is expected since K-fold CV provides a more robust and conservative estimate of model performance. The overall ranking of the models remained consistent

# <font color='#c62cce'> <b>4. PCA  </b></font>

## **a) Logistic Regression + PCA**

In [ ]:
# ============================================
# PCA + Logistic Regression
# ============================================

# Pipeline: Standardize -> PCA -> Logistic Regression
pipe_lr_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),  # keep 95% of the variance
    ("clf", LogisticRegression(
        class_weight="balanced",
        multi_class="multinomial",
        max_iter=1000,
        random_state=42
    ))
])

# Fit on training data (no SMOTE here)
pipe_lr_pca.fit(X_train, y_train)

# Evaluate on test set
results_lr_pca = evaluate_model_multiclass(pipe_lr_pca, X_test, y_test)
results_lr_pca

## **b) KNN + PCA**

In [ ]:
# ============================================
# PCA + KNN
# ============================================

# Pipeline: Standardize -> PCA -> KNN
pipe_knn_pca = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=0.95)),  # same setting for consistency
    ("knn", KNeighborsClassifier(n_neighbors=5))
])

pipe_knn_pca.fit(X_train, y_train)

results_knn_pca = evaluate_model_multiclass(pipe_knn_pca, X_test, y_test)
results_knn_pca

### **PCA – Results and Interpretation**

Applying PCA before Logistic Regression and KNN does not improve performance on our
dataset. For Logistic Regression, PCA yields almost identical results to the baseline,
with a very slight decrease in F1-macro and ROC-AUC. This is expected because Logistic
Regression already handles high-dimensional sparse data reasonably well once features
are standardized.

For KNN, PCA slightly worsens performance. This is also normal: KNN relies heavily on
the original distance relationships between samples, and PCA modifies the geometry of
the feature space, which can negatively affect nearest-neighbor search.

Overall, PCA does not provide any benefit for our models and is therefore not used in
the final optimized pipeline. Tree-based models (Random Forest, XGBoost, CatBoost)
do not require PCA, and linear models show no meaningful improvement from it.

# <font color='#c62cce'> <b>5. Hyperparameters tuning  </b></font>

## Why RandomizedSearchCV instead of GridSearchCV ?

We use RandomizedSearchCV rather than GridSearchCV because our models
(Random Forest and XGBoost) have a large number of hyperparameters.
Grid Search would require evaluating hundreds or thousands of parameter
combinations, which is computationally expensive, especially with a
5-fold Stratified Cross-Validation.

Randomized Search allows us to explore a wide hyperparameter space much
faster by sampling a limited number of random combinations. In practice,
RandomizedSearchCV often reaches similar or better performance than
GridSearchCV while reducing computation time dramatically. This makes it
the most efficient and realistic choice for our project.

## **a) Logistic regression + Hyperparameters tuning**

In [ ]:
# ============================================
# Logistic Regression - Hyperparameter Tuning
# ============================================

pipe_lr = Pipeline([
    ("scaler", StandardScaler()),
    ("clf", LogisticRegression(
        class_weight="balanced",
        multi_class="multinomial",
        max_iter=1000,
        random_state=42
    ))
])

param_lr = {
    "clf__C": [0.01, 0.1, 1, 10, 100],
    "clf__penalty": ["l2"],  # l1 would require saga solver
}

lr_search = RandomizedSearchCV(
    estimator=pipe_lr,
    param_distributions=param_lr,
    n_iter=5,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    random_state=42,
    verbose=1
)

lr_search.fit(X_train, y_train)

best_lr = lr_search.best_estimator_
print("Best Logistic Regression params:", lr_search.best_params_)

results_lr_tuned = evaluate_model_multiclass(best_lr, X_test, y_test)
results_lr_tuned

## **b) Decision Tree + Hyperparameter Tuning**

In [ ]:
# ============================================
# Decision Tree - Hyperparameter Tuning
# ============================================

dt_base = DecisionTreeClassifier(
    class_weight="balanced",
    random_state=42
)

param_dt = {
    "criterion": ["gini", "entropy"],
    "max_depth": [None, 5, 10, 20, 30],
    "min_samples_split": [2, 5, 10, 20],
    "min_samples_leaf": [1, 2, 4, 8]
}

dt_search = RandomizedSearchCV(
    estimator=dt_base,
    param_distributions=param_dt,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,
    random_state=42,
    verbose=1
)

dt_search.fit(X_train, y_train)

best_dt = dt_search.best_estimator_
print("Best Decision Tree params:", dt_search.best_params_)

results_dt_tuned = evaluate_model_multiclass(best_dt, X_test, y_test)
results_dt_tuned

## **c) Random Forest + Hyperparameter Tuning**

In [ ]:
# ============================================
# Random Forest - Hyperparameter Tuning
# ============================================

rf_base = RandomForestClassifier(
    class_weight="balanced",
    random_state=42
)

param_rf = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "criterion": ["gini", "entropy"],
    "max_features": ["sqrt", "log2"]
}

rf_search = RandomizedSearchCV(
    estimator=rf_base,
    param_distributions=param_rf,
    n_iter=20,
    scoring="f1_macro",
    cv=cv,
    n_jobs=1,      # keep 1 to avoid memory issues
    random_state=42,
    verbose=1
)

rf_search.fit(X_train, y_train)

best_rf = rf_search.best_estimator_
print("Best Random Forest params:", rf_search.best_params_)

results_rf_tuned = evaluate_model_multiclass(best_rf, X_test, y_test)
results_rf_tuned

## **d) XGBoost + Hyperparameter Tuning**

In [ ]:
cv_xgb = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)

# Base XGBoost model
xgb_base = XGBClassifier(
    objective="multi:softprob",
    num_class=3,
    eval_metric="mlogloss",
    tree_method="hist",   # histogram-based, plus rapide
    random_state=42,
    n_jobs=-1              # utilise tous les cores dispo
)

# Hyperparameter space (réduit mais raisonnable)
param_xgb = {
    "n_estimators": [100, 200, 300],      # moins que 600 → beaucoup plus rapide
    "max_depth": [3, 5, 7],
    "learning_rate": [0.05, 0.1],
    "subsample": [0.7, 0.9],
    "colsample_bytree": [0.7, 0.9]
}

xgb_search = RandomizedSearchCV(
    estimator=xgb_base,
    param_distributions=param_xgb,
    n_iter=10,              # 10 combinaisons aléatoires → suffisant pour un projet
    scoring="f1_macro",
    cv=cv_xgb,
    n_jobs=-1,              # parallélisation sur tous les cores
    random_state=42,
    verbose=1
)

xgb_search.fit(X_train_xgb, y_train)

best_xgb = xgb_search.best_estimator_
print("Best XGBoost params:", xgb_search.best_params_)

results_xgb_tuned = evaluate_model_multiclass(best_xgb, X_test_xgb, y_test)
results_xgb_tuned


# <font color='#c62cce'> <b>6. Ensemble learning  </b></font>

In [ ]:
# ============================================
# Ensemble 1: Bagging with Decision Tree
# ============================================

bagging_dt = BaggingClassifier(
    estimator=best_dt,        # or dt if you only have the baseline tree
    n_estimators=50,
    max_samples=0.8,
    max_features=1.0,
    bootstrap=True,
    n_jobs=-1,
    random_state=42
)

bagging_dt.fit(X_train, y_train)

results_bagging_dt = evaluate_model_multiclass(bagging_dt, X_test, y_test)
results_bagging_dt

In [ ]:
# ============================================
# Ensemble 2: Soft Voting (LR + RF + XGB)
# ============================================

# If you don't have best_lr (tuned), replace 'best_lr' by your baseline 'log_reg'
voting_clf = VotingClassifier(
    estimators=[
        ("lr", best_lr),   # or log_reg
        ("rf", best_rf),
        ("xgb", best_xgb)
    ],
    voting="soft",
    n_jobs=-1
)

# Use the cleaned feature names version (same numeric values, just safer for XGBoost)
voting_clf.fit(X_train_xgb, y_train)

results_voting = evaluate_model_multiclass(voting_clf, X_test_xgb, y_test)
results_voting

In [ ]:
# ============================================
# Ensemble 3: Stacking Classifier
# ============================================

stack_clf = StackingClassifier(
    estimators=[
        ("rf", best_rf),
        ("dt", best_dt),
        ("xgb", best_xgb)
    ],
    final_estimator=LogisticRegression(
        class_weight="balanced",
        max_iter=1000,
        multi_class="multinomial",
        random_state=42
    ),
    n_jobs=-1,
    passthrough=False
)

stack_clf.fit(X_train_xgb, y_train)

results_stack = evaluate_model_multiclass(stack_clf, X_test_xgb, y_test)
results_stack


# <font color='#c62cce'> <b>7. Advanced model  </b></font>

In [ ]:
# ============================================ 
# STEP 9 — CatBoost (advanced model) 
# ============================================ 
 
cat_clf = CatBoostClassifier( 
   loss_function="MultiClass", 
   eval_metric="TotalF1", 
   iterations=500, 
   depth=6, 
   learning_rate=0.1, 
   random_seed=42, 
   verbose=False 
) 
 
cat_clf.fit(X_train, y_train) 
 
results_cat = evaluate_model_multiclass(cat_clf, X_test, y_test) 
results_cat

# <font color='#c62cce'> <b>8. Confusion matrix and final comparison   </b></font>

In [ ]:
import sys
print(sys.executable)
